# Сравнение сборщиков с эталоном — человек PRJEB30386

Один смешанный образец IGH+IGK+IGL (4 библиотеки 5′MTPX).

**Эталон** — наблюдаемые последовательности V…J шаблонов симуляции (`00_primary_truth`, интервал `vj_start…vj_end`) с сохранёнными SHM; CDR3 эталона — AIRR `junction` исходных ридов.

**Recall (по шаблонам).** Каждая эталонная V…J выравнивается minimap2 на контиги сборщика; берётся лучшее выравнивание.

| класс | условие |
|---|---|
| `exact_full` | вся V…J входит в контиг подстрокой (на любой цепи) — **основная метрика** |
| `one_diff_full` | вся V…J, 1 отличие (основания, обрезанные minimap2 на концах, сравниваются с контигом) |
| `near_full` | ≥95% V…J, идентичность ≥98% |
| `partial` | есть выравнивание, но хуже |
| `missing` | выравнивания нет |

**Проверенные шаблоны.** Recall, классы восстановимости и CDR3 считаются по шаблонам `verified=True` (человек — V…J подтверждена консенсусом UMI, мышь — ≥2 рида после схлопывания ошибок). Непроверенные шаблоны есть в библиотеке, но их последовательность может нести ошибку секвенирования исходных данных. Контиг, лучше всего совпавший с непроверенным шаблоном, получает класс precision `unverified_match` и ошибкой не считается. Recall по непроверенным шаблонам показан отдельной строкой `UNVERIFIED` (часть из них — ошибки исходных данных).

В эталоне много клональных родственников, отличающихся 1–7 SHM, поэтому мягкие классы засчитывают шаблону контиг родственника. **Отрицательный контроль** — шаблоны класса восстановимости `no_reads` (их риды не попали в FASTQ, собрать их нельзя): доля «восстановленных» среди них показывает частоту ложных совпадений на каждом уровне строгости.

**CDR3 (по клонотипам = образец + локус + junction).** Клонотип найден, если его junction целиком содержится в каком-либо контиге образца. Контроль — клонотипы, ни один шаблон которых не прочитан.

**Precision (по контигам).** Каждый контиг выравнивается на эталонные V…J: `exact_full` — содержит проверенную эталонную V…J подстрокой; `near_full` — ≥95% V…J при идентичности ≥98%; `correct_partial` — ≥100 нт при идентичности ≥98%; `incorrect` — ниже (ошибки, химеры); `unmapped`.

**Уникальные V…J без зоны праймера.** Эталон — уникальные последовательности V…J без первых `TRIM5` нт (мышь 20, человек 0): у мыши эту зону задаёт FR1-праймер, севший на молекулу, и одна молекула записывалась несколькими «шаблонами», различающимися только праймером. Такие шаблоны объединяются (колонка `members`). Результаты — в `benchmark_unique_trim<TRIM5>/`.

**Структура контига** (`structure`, по всем выравниваниям контига на эталон): `exact_verified` / `exact_unverified` — содержит эталонную V…J (проверенную / непроверенную) подстрокой; `exact_fragment` — у лучшего выравнивания нет отличий, но целой эталонной V…J в контиге нет (правильный кусок, контиг `phase_correct` неполной длины); `chimera` — 5′- и 3′-части (по 60%) точно совпадают с разными эталонными V…J; `mosaic` — каждое отличие от лучшего эталона есть у другого **проверенного** эталона (перемешаны мутации родственников); `unverified_variant` — отличия объясняются только непроверенными эталонами; `novel_error` — есть отличие, которого нет ни у одного эталона; `other` — нет выравнивания с идентичностью ≥98%. Во входных данных симуляции химер нет, поэтому `chimera` и `mosaic` создаёт сборщик.

**Поддержка контига** (`support`): TRUST4 — число ридов (`consensus_count`), rnaSPAdes — k-мерное покрытие, Trinity — нет. `assemblers_structure_by_support.tsv` — структура по классам поддержки; `assemblers_support_thresholds.tsv` — сколько контигов остаётся и какая доля проверенных V…J собрана точно, если отбросить контиги с поддержкой ниже порога (V…J считается собранной точно, если хотя бы один содержащий её контиг проходит порог).

**Блоки фазы** (как оценка фазирования в геномике; считаются для каждого контига в пролёте его лучшего выравнивания на эталон). Контиг разбивается на минимальное число **блоков** — кусков, точно совпадающих с отрезком одной эталонной V…J. `switches` — стыки соседних блоков без зазора: контиг переходит с одной молекулы на другую (ложная склейка, мозаика; оценка снизу). `novel_positions` — позиции, которых нет ни у одного эталона (ошибки). `phase_correct` — весь пролёт контига — один точный отрезок одной молекулы. Сводки (`assemblers_phase_blocks.tsv`): `phase_correct_frac` — доля выровненных контигов без единой ложной склейки и ошибки (**точность блоков**); `contigs_with_switch_frac` и `switches_per_kb` — **ложные склейки**; `novel_per_kb` — ошибки; по эталону `exact_block_frac` — доля длины V…J, которую целиком покрывает один `phase_correct` контиг (1.0 ≈ `exact_full`), сводки `exact_block_mean` и `exact_block_ge{50,80,100}_frac` — **покрытие эталона правильными блоками**. Отрезок, общий для нескольких родственников, засчитывается каждому из них. Метрика вознаграждает сборщик, который там, где фаза не определена, выдаёт правильные куски, а не мозаики.

## Окружение и параметры

In [ ]:
import os, sys, shutil, subprocess, time, gzip, json
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]

DATASET = "PRJEB30386"
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_umicons_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["PRJEB30386_all_chains"]
THREADS = int(os.environ.get("BCR_THREADS", 20))
# Первые TRIM5 нт V…J эталона не сравниваются: у мыши это зона FR1-праймера, её задаёт праймер, а не молекула
# (19 FR1-праймеров Greiff 2014; 16 463 проверенных шаблона ERR346600 дают лишь 1 766 уникальных V…J без неё).
TRIM5 = int(os.environ.get("BCR_TRUTH_TRIM_5P", 0))
FORCE = os.environ.get("BCR_BENCH_FORCE") == "1"   # пересчитать готовые per_template/per_contig

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
DATASET_DIR = ROOT / "results" / DATASET
BRANCH_DIR = DATASET_DIR / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
ASM_DIR = BRANCH_DIR / "assemblers"

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

def iter_fasta(path):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    """Нормализованный FASTA контигов для сравнения с эталоном."""
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

print("ROOT:", ROOT); print("BRANCH:", BRANCH_DIR); print("samples:", SAMPLES, "threads:", THREADS)


## Функции сравнения

In [ ]:
import csv, gzip, re, subprocess
from collections import defaultdict, Counter
from pathlib import Path


def open_text(path):
    path = str(path)
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path)


def iter_fasta(path):
    name, chunks = None, []
    with open_text(path) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)


CLASS_RANK = ["full_coverage_read_overlap", "full_coverage_mate_linked_only", "full_coverage_unlinked",
              "partial_read_coverage", "no_reads", "not_classified"]


def load_truth(branch_dir, sample, trim5=0):
    """Эталон: уникальные V…J (локус + последовательность) без первых `trim5` нт — зоны праймера.

    Шаблоны, различающиеся только в этой зоне (разные FR1-праймеры / вырожденные позиции на одной молекуле),
    объединяются в одну запись: проверена, если проверен хоть один шаблон; класс восстановимости — лучший;
    копии суммируются; `members` — число объединённых шаблонов.
    """
    meta = {r["template_id"]: r for r in csv.DictReader(
        open(branch_dir / "00_primary_truth" / f"{sample}_template_qc.tsv"), delimiter="\t")}
    cls = {r["template_id"]: r["reconstructability"] for r in csv.DictReader(
        open(branch_dir / "qc" / f"{sample}_template_reconstructability.tsv"), delimiter="\t")}
    rank = {c: i for i, c in enumerate(CLASS_RANK)}
    groups = {}
    for tid, seq in iter_fasta(branch_dir / "00_primary_truth" / f"{sample}_templates.fasta"):
        m = meta[tid]
        rec = {"tid": tid, "locus": m["locus"], "vj": seq[int(m["vj_start"]) + trim5:int(m["vj_end"])],
               "class": cls.get(tid, "not_classified"), "verified": m["verified"] == "True",
               "junction": m["junction"], "n": int(m.get("unique_umis") or m.get("observed_multiplicity") or 1)}
        groups.setdefault((rec["locus"], rec["vj"]), []).append(rec)
    truth = {}
    for members in groups.values():
        rep = min(members, key=lambda r: (not r["verified"], rank.get(r["class"], len(rank)), r["tid"]))
        n = sum(r["n"] for r in members)
        truth[rep["tid"]] = {"locus": rep["locus"], "vj": rep["vj"], "junction": rep["junction"],
                             "class": min((r["class"] for r in members), key=lambda c: rank.get(c, len(rank))),
                             "verified": any(r["verified"] for r in members), "members": len(members),
                             "copies": "1" if n == 1 else ("2-4" if n < 5 else "5+")}
    return truth


COMP = str.maketrans("ACGTN", "TGCAN")


def revcomp(s):
    return s.translate(COMP)[::-1]


def contained(queries, refs, k=31):
    """Точное вхождение: id запроса -> множество id последовательностей `refs`, содержащих запрос целиком
    (на любой цепи). Кандидаты ищутся по первому k-меру запроса, затем проверяется всё вхождение."""
    seeds = defaultdict(list)
    for qid, q in queries.items():
        if len(q) >= k:
            seeds[q[:k]].append(qid)
    short = {qid: q for qid, q in queries.items() if len(q) < k}
    hits = defaultdict(set)
    for rid, r in refs.items():
        for strand in (r, revcomp(r)):
            for i in range(len(strand) - k + 1):
                for qid in seeds.get(strand[i:i + k], ()):
                    if strand.startswith(queries[qid], i):
                        hits[qid].add(rid)
            for qid, q in short.items():
                if q in strand:
                    hits[qid].add(rid)
    return hits


def end_mismatches(h, qseq, tseq):
    """Отличия в концах запроса, которые minimap2 обрезал (soft clip), если контиг их перекрывает.
    Возвращает число несовпадающих оснований или None, если контиг не доходит до конца запроса."""
    t = tseq if h["strand"] == "+" else revcomp(tseq)
    ts, te = (h["ts"], h["te"]) if h["strand"] == "+" else (h["tlen"] - h["te"], h["tlen"] - h["ts"])
    c5, c3 = h["qs"], h["qlen"] - h["qe"]
    if ts < c5 or len(t) - te < c3:
        return None
    left = sum(a != b for a, b in zip(qseq[:c5], t[ts - c5:ts]))
    right = sum(a != b for a, b in zip(qseq[h["qe"]:], t[te:te + c3]))
    return left + right


def minimap2_paf(reference, query, out_paf, threads):
    """Выравнивания с базовым разрешением (PAF с тегом NM) запросов на референс."""
    cmd = ["minimap2", "-c", "-x", "sr", "-N", "50", "-p", "0.5", "--secondary=yes",
           "-t", str(threads), str(reference), str(query)]
    with open(out_paf, "w") as out:
        subprocess.run(cmd, stdout=out, stderr=subprocess.DEVNULL, check=True)


def iter_paf(path):
    with open(path) as h:
        for line in h:
            f = line.rstrip("\n").split("\t")
            nm = next((int(x[5:]) for x in f[12:] if x.startswith("NM:i:")), 0)
            yield {"q": f[0], "qlen": int(f[1]), "qs": int(f[2]), "qe": int(f[3]),
                   "strand": f[4], "t": f[5], "tlen": int(f[6]), "ts": int(f[7]), "te": int(f[8]),
                   "match": int(f[9]), "block": int(f[10]), "nm": nm}


def recall_class(hit):
    """Класс эталонной V–J (запрос), выровненной на контиги (референс).

    exact_full — вся V–J без единой замены/инделя; one_diff_full — вся V–J, 1 отличие (основания, обрезанные
    minimap2 на концах V–J, сравниваются с контигом и считаются отличиями — `end_mismatches`);
    near_full — >=95% V–J с идентичностью >=98% (мягкий порог, путает клональных родственников);
    partial — остальное с выравниванием; missing — выравнивания нет.
    """
    if hit is None:
        return "missing"
    cov = (hit["qe"] - hit["qs"]) / hit["qlen"]
    ident = hit["match"] / hit["block"]
    clipped = hit.get("end_mismatches")
    edits = hit["nm"] if cov == 1.0 else (hit["nm"] + clipped if clipped is not None else None)
    if edits == 0:
        return "exact_full"
    if edits == 1:
        return "one_diff_full"
    if cov >= 0.95 and ident >= 0.98:
        return "near_full"
    return "partial"


def precision_class(hit):
    """Класс контига (запрос), выровненного на эталонные V–J (референс)."""
    if hit is None:
        return "unmapped"
    ident = hit["match"] / hit["block"]
    tcov = (hit["te"] - hit["ts"]) / hit["tlen"]
    if tcov == 1.0 and hit["nm"] == 0:
        return "exact_full"
    if ident >= 0.98 and tcov >= 0.95:
        return "near_full"
    if ident >= 0.98 and hit["block"] >= 100:
        return "correct_partial"
    return "incorrect"


def best_by_query(paf_path, preferred=frozenset()):
    """Лучшее выравнивание каждого запроса; при равенстве предпочитается референс из `preferred`."""
    best, targets = {}, defaultdict(set)
    for h in iter_paf(paf_path):
        targets[h["q"]].add(h["t"])
        key = (h["match"] - h["nm"], h["qe"] - h["qs"], h["t"] in preferred)
        if h["q"] not in best or key > best[h["q"]][0]:
            best[h["q"]] = (key, h)
    return {q: v[1] for q, v in best.items()}, targets


def cs_hits(reference, query, threads):
    """Все выравнивания запросов (контигов) на эталон с позициями отличий в координатах запроса (minimap2 --cs)."""
    cmd = ["minimap2", "-c", "--cs", "-x", "sr", "-N", "50", "-p", "0.5", "--secondary=yes",
           "-t", str(threads), str(reference), str(query)]
    hits = defaultdict(list)
    for line in subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.splitlines():
        f = line.split("\t")
        h = {"q": f[0], "qlen": int(f[1]), "qs": int(f[2]), "qe": int(f[3]), "strand": f[4], "t": f[5],
             "tlen": int(f[6]), "ts": int(f[7]), "te": int(f[8]), "match": int(f[9]), "block": int(f[10]),
             "nm": next(int(x[5:]) for x in f[12:] if x.startswith("NM:i:")),
             "cs": next(x[5:] for x in f[12:] if x.startswith("cs:Z:"))}
        h["diffs"] = query_diffs(h)
        hits[h["q"]].append(h)
    return hits


def query_diffs(h):
    """Позиции отличий в координатах запроса. cs идёт по референсу: '*xy' замена, '+seq' лишние основания
    запроса, '-seq' основания референса, которых нет в запросе. Для минус-цепи координаты зеркалятся."""
    minus = h["strand"] == "-"
    pos, out = (h["qlen"] - h["qe"]) if minus else h["qs"], set()
    for op, val in re.findall(r"([:*+\-])([0-9]+|[a-z]+)", h["cs"]):
        if op == ":":
            pos += int(val)
        elif op == "*":
            out.add(pos); pos += 1
        elif op == "+":
            out.update(range(pos, pos + len(val))); pos += len(val)
        else:
            out.add(pos)
    return {h["qlen"] - 1 - p for p in out} if minus else out


def contig_structure(hits, parts_exact, verified=frozenset(), exact_targets=frozenset()):
    """exact_verified / exact_unverified — контиг содержит эталонную V…J (проверенную / непроверенную) целиком без
    отличий (`exact_targets` — эталоны, входящие в контиг подстрокой); exact_fragment — у лучшего выравнивания нет
    отличий, но целой эталонной V…J в контиге нет (правильный кусок; это контиги phase_correct неполной длины); chimera — 5'- и 3'-части (по 60%) точно совпадают с РАЗНЫМИ эталонными V…J; mosaic — каждое отличие
    от лучшего эталона есть у другого ПРОВЕРЕННОГО эталона (перемешаны мутации родственников);
    unverified_variant — отличия объясняются только непроверенными эталонами (среди них ошибки исходных данных);
    novel_error — есть отличие, которого нет ни у одного эталона; other — нет выравнивания с идентичностью >= 98%."""
    if exact_targets:
        return "exact_verified" if exact_targets & verified else "exact_unverified"
    if not hits:
        return "unmapped"
    p5, p3 = parts_exact
    if p5 and p3 and p5 != p3:
        return "chimera"
    good = [h for h in hits if h["match"] / h["block"] >= 0.98]
    if not good:
        return "other"
    best = max(good, key=lambda h: (h["match"] - h["nm"], h["qe"] - h["qs"]))
    if not best["diffs"]:
        return "exact_fragment"

    def explained(pool):
        return all(any(o is not best and o["qs"] <= p < o["qe"] and p not in o["diffs"] for o in pool)
                   for p in best["diffs"])

    if explained([o for o in good if o["t"] in verified]):
        return "mosaic"
    return "unverified_variant" if explained(good) else "novel_error"


def contig_support(asm_dir, tool, sample):
    """Поддержка контига: TRUST4 — consensus_count (число ридов) из AIRR; rnaSPAdes — k-мерное покрытие `cov`
    из имени контига; Trinity — нет (пустой словарь). Ключ — id контига в contigs.fasta."""
    native = {}
    for line in open(asm_dir / tool / sample / "contigs.fasta"):
        if line.startswith(">"):
            p = line[1:].split()
            native[p[0]] = p[1] if len(p) > 1 else ""
    if tool == "trust4":
        counts = {r["sequence_id"]: float(r["consensus_count"] or 0) for r in csv.DictReader(
            open(asm_dir / tool / sample / f"TRUST4_{sample}_airr.tsv"), delimiter="\t")}
        return {c: counts[n] for c, n in native.items() if n in counts}
    if tool == "rnaspades":
        return {c: float(m.group(1)) for c, n in native.items() if (m := re.search(r"_cov_([0-9.]+)", n))}
    return {}


def chimera_parts(contigs, exact_ids, truth_fa, workdir, threads):
    """Лучшее точное совпадение 5'- и 3'-частей (по 60%) контигов без точной копии эталона."""
    parts = workdir / "contig_parts.fasta"
    with open(parts, "w") as out:
        for cid, s in contigs.items():
            if cid not in exact_ids and len(s) >= 200:
                k = int(len(s) * 0.6)
                out.write(f">{cid}|5\n{s[:k]}\n>{cid}|3\n{s[-k:]}\n")
    res = defaultdict(lambda: [None, None])
    for q, hs in cs_hits(truth_fa, parts, threads).items():
        ex = [h["t"] for h in hs if h["nm"] == 0 and h["qe"] - h["qs"] == h["qlen"]]
        if ex:
            cid, side = q.rsplit("|", 1)
            res[cid][0 if side == "5" else 1] = sorted(ex)[0]
    return res



def exact_intervals(h):
    """Отрезки запроса [s, e), на которых выравнивание h совпадает с эталоном без отличий."""
    out, start = [], h["qs"]
    for p in sorted(d for d in h["diffs"] if h["qs"] <= d < h["qe"]):
        if p > start:
            out.append((start, p))
        start = p + 1
    if h["qe"] > start:
        out.append((start, h["qe"]))
    return out


def phase_blocks(hits):
    """Разбиение контига на блоки фазы в пролёте лучшего выравнивания (как в contig_structure).

    Блок — кусок контига, точно совпадающий с отрезком одной эталонной V…J. Берётся минимальное число блоков
    (жадное покрытие отрезками). switches — стыки соседних блоков без зазора: контиг переходит с одной молекулы
    на другую (ложная склейка; оценка снизу). novel — позиции, которых нет ни у одного эталона.
    phase_correct — весь пролёт = один точный отрезок одной молекулы. credit — эталоны, у которых такой
    контиг точно покрывает долю длины (для phase_correct; родственники с тем же отрезком получают его все)."""
    good = [h for h in hits if h["match"] / h["block"] >= 0.98]
    if not good:
        return None
    best = max(good, key=lambda h: (h["match"] - h["nm"], h["qe"] - h["qs"]))
    a, b = best["qs"], best["qe"]
    ivs = [iv for h in good for iv in exact_intervals(h)]
    p, blocks, switches, novel, longest, touching = a, 0, 0, 0, 0, False
    while p < b:
        e = max((x2 for x1, x2 in ivs if x1 <= p < x2), default=None)
        if e is None:
            nxt = min((x1 for x1, _ in ivs if p < x1 < b), default=b)
            novel += nxt - p
            p, touching = nxt, False
            continue
        e = min(e, b)
        blocks += 1
        switches += touching
        longest = max(longest, e - p)
        p, touching = e, True
    ok = blocks == 1 and novel == 0
    credit = {}
    if ok:
        for h in good:
            if h["qs"] <= a and h["qe"] >= b and not any(a <= d < b for d in h["diffs"]):
                credit[h["t"]] = max(credit.get(h["t"], 0), min(1.0, (b - a) / h["tlen"]))
    return {"aligned_span": b - a, "blocks": blocks, "switches": switches, "novel_positions": novel,
            "longest_block": longest, "phase_correct": ok, "credit": credit}


def benchmark(truth, contigs, workdir, threads, native_cdr3=None):
    """Строки по шаблонам и по контигам. `contigs`: id -> последовательность."""
    workdir.mkdir(parents=True, exist_ok=True)
    truth_fa, contig_fa = workdir / "truth_vj.fasta", workdir / "contigs.fasta"
    with open(truth_fa, "w") as h:
        for tid, t in truth.items():
            h.write(f">{tid}\n{t['vj']}\n")
    with open(contig_fa, "w") as h:
        for cid, s in contigs.items():
            h.write(f">{cid}\n{s}\n")

    minimap2_paf(contig_fa, truth_fa, workdir / "truth_to_contigs.paf", threads)
    minimap2_paf(truth_fa, contig_fa, workdir / "contigs_to_truth.paf", threads)

    best_t, hit_contigs = best_by_query(workdir / "truth_to_contigs.paf")
    exact_of = contained({tid: t["vj"] for tid, t in truth.items()}, contigs)
    junctions = {t["junction"] for t in truth.values() if t.get("junction")}
    junction_found = contained({j: j for j in junctions}, contigs, min([15] + [len(j) for j in junctions]))
    template_rows = []
    for tid, t in truth.items():
        h = best_t.get(tid)
        if h and h["qe"] - h["qs"] < h["qlen"]:
            h = {**h, "end_mismatches": end_mismatches(h, t["vj"], contigs[h["t"]])}
        exact = sorted(exact_of.get(tid, ()))
        rc = recall_class(h)
        j = t.get("junction")
        cdr3 = "no_truth_junction" if not j else ("found" if junction_found.get(j) else "not_found")
        row = {"template_id": tid, "locus": t["locus"], "reconstructability": t["class"], "copies": t["copies"],
               "verified": t["verified"],
               "junction": j or "",
               "recall": "exact_full" if exact else ("one_diff_full" if rc == "exact_full" else rc),
               "best_contig": (h["t"] if h and h["t"] in exact else exact[0]) if exact else (h["t"] if h else ""),
               "exact_contigs": ";".join(exact),
               "vj_coverage": round((h["qe"] - h["qs"]) / h["qlen"], 4) if h else 0,
               "identity": round(h["match"] / h["block"], 4) if h else 0,
               "edits": h["nm"] if h else "",
               "cdr3_in_contig": cdr3}
        if native_cdr3 is not None:
            row["cdr3_native"] = "no_truth_junction" if not j else ("found" if j in native_cdr3 else "not_found")
        template_rows.append(row)

    verified = frozenset(tid for tid, t in truth.items() if t["verified"])
    best_c, _ = best_by_query(workdir / "contigs_to_truth.paf", verified)
    hits = cs_hits(truth_fa, contig_fa, threads)
    exact_in = defaultdict(set)                     # контиг -> эталоны, входящие в него подстрокой
    for tid, cs in exact_of.items():
        for c in cs:
            exact_in[c].add(tid)
    exact_ids = set(exact_in)
    parts = chimera_parts(contigs, exact_ids, truth_fa, workdir, threads)
    contig_rows, credit = [], defaultdict(float)
    for cid, s in contigs.items():
        h = best_c.get(cid)
        if cid in exact_in:
            cls = "exact_full" if exact_in[cid] & verified else "unverified_match"
        else:
            cls = precision_class(h)
            cls = "near_full" if cls == "exact_full" else cls
            if h and h["t"] not in verified and cls not in ("incorrect", "unmapped"):
                cls = "unverified_match"
        pb = phase_blocks(hits.get(cid, [])) or {}
        for t, f in pb.get("credit", {}).items():
            credit[t] = max(credit[t], f)
        contig_rows.append({"contig_id": cid, "length": len(s), "precision": cls,
                            "structure": contig_structure(hits.get(cid, []), parts.get(cid, (None, None)), verified,
                                                         frozenset(exact_in.get(cid, ()))),
                            "best_template": h["t"] if h else "",
                            "identity": round(h["match"] / h["block"], 4) if h else 0,
                            **{k: pb.get(k, "") for k in ("aligned_span", "blocks", "switches", "novel_positions",
                                                          "longest_block", "phase_correct")}})
    for tid in exact_of:                            # эталон целиком в контиге — блок на всю длину
        credit[tid] = 1.0
    for r in template_rows:
        r["exact_block_frac"] = round(credit.get(r["template_id"], 0.0), 4)
    return template_rows, contig_rows


RECALL = ["exact_full", "one_diff_full", "near_full", "partial", "missing"]
PRECISION = ["exact_full", "near_full", "correct_partial", "unverified_match", "incorrect", "unmapped"]
STRUCTURE = ["exact_verified", "exact_unverified", "exact_fragment", "mosaic", "unverified_variant", "novel_error", "chimera", "other", "unmapped"]


def summarize(template_rows, contig_rows, group_keys=("reconstructability",)):
    """Recall по проверенным шаблонам в группах + строка UNVERIFIED (все непроверенные) + precision."""
    groups = defaultdict(list)
    for r in template_rows:
        if str(r["verified"]) != "True":
            groups[("UNVERIFIED",) * len(group_keys)].append(r)
            continue
        groups[tuple(r[k] for k in group_keys)].append(r)
        groups[("ALL",) * len(group_keys)].append(r)
    out = []
    for g, rows in sorted(groups.items()):
        n = len(rows)
        c = Counter(r["recall"] for r in rows)
        row = {**dict(zip(group_keys, g)), "templates": n,
               **{f"{k}_frac": round(c[k] / n, 4) for k in RECALL}}
        fr = [float(r["exact_block_frac"]) for r in rows if str(r.get("exact_block_frac", "")) != ""]
        if fr:
            row.update(exact_block_mean=round(sum(fr) / len(fr), 4),
                       **{f"exact_block_ge{int(t * 100)}_frac": round(sum(x >= t for x in fr) / len(fr), 4)
                          for t in (0.5, 0.8, 1.0)})
        out.append(row)
    pc = Counter(r["precision"] for r in contig_rows)
    sc = Counter(r.get("structure", "") for r in contig_rows)
    precision = {"contigs": len(contig_rows),
                 **{f"{k}_frac": round(pc[k] / max(1, len(contig_rows)), 4) for k in PRECISION},
                 **{f"structure_{k}_frac": round(sc[k] / max(1, len(contig_rows)), 4) for k in STRUCTURE}}
    al = [r for r in contig_rows if str(r.get("aligned_span", "")) not in ("", "0")]
    if al:
        span = sum(int(r["aligned_span"]) for r in al)
        precision.update(
            block_aligned_contigs=len(al),
            phase_correct_frac=round(sum(str(r["phase_correct"]) == "True" for r in al) / len(al), 4),
            contigs_with_switch_frac=round(sum(int(r["switches"]) > 0 for r in al) / len(al), 4),
            switches_per_kb=round(1000 * sum(int(r["switches"]) for r in al) / span, 3),
            novel_per_kb=round(1000 * sum(int(r["novel_positions"]) for r in al) / span, 3))
    return out, precision


def summarize_clonotypes(template_rows):
    """CDR3 на уровне клонотипов (локус + junction): клонотип «прочитан», если у любого его
    шаблона есть риды; отрицательный контроль — клонотипы, ни один шаблон которых не прочитан.
    Учитываются только проверенные шаблоны."""
    clon = defaultdict(list)
    for r in template_rows:
        if r["junction"] and str(r["verified"]) == "True":
            clon[(r.get("sample", ""), r["locus"], r["junction"])].append(r)
    groups = defaultdict(list)
    for key, rows in clon.items():
        state = "no_reads_control" if all(r["reconstructability"] == "no_reads" for r in rows) else "sequenced"
        rec = {"in_contig": any(r["cdr3_in_contig"] == "found" for r in rows),
               "native": any(r.get("cdr3_native") == "found" for r in rows)}
        groups[(key[1], state)].append(rec); groups[("ALL", state)].append(rec)
    out = []
    for (locus, state), recs in sorted(groups.items()):
        row = {"locus": locus, "clonotype_state": state, "clonotypes": len(recs),
               "cdr3_in_contig_frac": round(sum(x["in_contig"] for x in recs) / len(recs), 4)}
        if "cdr3_native" in template_rows[0]:
            row["cdr3_native_frac"] = round(sum(x["native"] for x in recs) / len(recs), 4)
        out.append(row)
    return out


def write_tsv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=list(rows[0]), delimiter="\t")
        w.writeheader()
        w.writerows(rows)


# самопроверка классификаторов
_h = {"qs": 0, "qe": 100, "qlen": 100, "match": 100, "block": 100, "nm": 0, "ts": 0, "te": 100, "tlen": 100}
assert recall_class(_h) == "exact_full" and precision_class(_h) == "exact_full"
assert recall_class({**_h, "match": 99, "nm": 1}) == "one_diff_full"
assert recall_class({**_h, "match": 98, "nm": 2}) == "near_full"
assert recall_class({**_h, "qe": 50}) == "partial" and recall_class(None) == "missing"
assert precision_class({**_h, "match": 90, "nm": 10}) == "incorrect" and precision_class(None) == "unmapped"

# самопроверка структуры контигов
_q = {"qlen": 10, "qs": 0, "qe": 10, "strand": "+", "cs": ":3*ag:6"}
assert query_diffs(_q) == {3} and query_diffs({**_q, "strand": "-", "cs": ":3*ag:6"}) == {6}
_b = {"t": "A", "qs": 0, "qe": 10, "ts": 0, "te": 10, "tlen": 10, "match": 99, "block": 100, "nm": 1, "diffs": {3}}
_o = {"t": "B", "qs": 0, "qe": 10, "ts": 0, "te": 10, "tlen": 10, "match": 98, "block": 100, "nm": 2, "diffs": {7}}
assert contig_structure([_b, _o], (None, None), {"B"}) == "mosaic"
assert contig_structure([_b, _o], (None, None), set()) == "unverified_variant"
assert contig_structure([_b, {**_o, "diffs": {3}}], (None, None), {"B"}) == "novel_error"
assert contig_structure([_b], ("t1", "t2")) == "chimera"
assert contig_structure([_b], (None, None), set(), frozenset({"A"})) == "exact_unverified"

# самопроверка блоков фазы
_A = {"t": "A", "qs": 0, "qe": 100, "tlen": 100, "match": 99, "block": 100, "nm": 1, "diffs": {60}}
_B = {"t": "B", "qs": 0, "qe": 100, "tlen": 100, "match": 99, "block": 100, "nm": 1, "diffs": {20}}
assert exact_intervals(_A) == [(0, 60), (61, 100)]
_pb = phase_blocks([_A, _B])              # мозаика: до 60 — B, дальше — A
assert (_pb["blocks"], _pb["switches"], _pb["novel_positions"], _pb["phase_correct"]) == (2, 1, 0, False)
_pb = phase_blocks([_A])                  # одна ошибка без объяснения
assert (_pb["blocks"], _pb["switches"], _pb["novel_positions"]) == (2, 0, 1)
_pb = phase_blocks([{**_A, "nm": 0, "match": 100, "diffs": set()}, _B])
assert _pb["phase_correct"] and _pb["credit"] == {"A": 1.0} and phase_blocks([]) is None

# самопроверка точных вхождений, обрезанных концов и класса exact_fragment
assert contained({"a": "ACGTACGTAC", "b": "TTTTTGGGGG"}, {"c1": "NNACGTACGTACNN", "c2": "CCCCCAAAAA"}, 5) == {"a": {"c1"}, "b": {"c2"}}
_h = {"qs": 1, "qe": 10, "qlen": 10, "strand": "+", "ts": 3, "te": 12, "tlen": 15}
assert end_mismatches(_h, "AACGTACGTA", "GGTTACGTACGTAGG") == 1          # первое основание отличается от контига
assert end_mismatches({**_h, "ts": 0}, "AACGTACGTA", "ACGTACGTAGGGGGG") is None   # контиг не доходит до начала
assert recall_class({"qs": 1, "qe": 10, "qlen": 10, "match": 9, "block": 9, "nm": 0, "end_mismatches": 1}) == "one_diff_full"
assert contig_structure([{**_b, "nm": 0, "diffs": set()}], (None, None), {"A"}) == "exact_fragment"
assert contig_structure([_b], (None, None), {"A"}, frozenset({"A"})) == "exact_verified"


## Доступные результаты сборщиков

In [ ]:
TOOLS = ["trust4", "mixcr", "rnaspades", "trinity", "vdjer"]
BENCH_DIR = BRANCH_DIR / f"benchmark_unique_trim{TRIM5}"
available = {t: [s for s in SAMPLES if (ASM_DIR / t / s / "contigs.fasta").exists()] for t in TOOLS}
print({t: len(s) for t, s in available.items()})


## Сравнение с эталоном

In [ ]:
import csv
all_rows, all_contigs, precision_rows = {}, {}, []
for tool, samples in available.items():
    for sample in samples:
        out = BENCH_DIR / tool / sample
        if (out / "per_template.tsv").exists() and not FORCE:
            rows = list(csv.DictReader(open(out / "per_template.tsv"), delimiter="\t"))
            crow = list(csv.DictReader(open(out / "per_contig.tsv"), delimiter="\t"))
        else:
            truth = load_truth(BRANCH_DIR, sample, TRIM5)
            contigs = dict(iter_fasta(ASM_DIR / tool / sample / "contigs.fasta"))
            rows, crow = benchmark(truth, contigs, out / "work", THREADS)
            write_tsv(out / "per_template.tsv", rows); write_tsv(out / "per_contig.tsv", crow)
        supp = contig_support(ASM_DIR, tool, sample)
        for r in crow:
            r["support"] = supp.get(r["contig_id"], "")
        write_tsv(out / "per_contig.tsv", crow)
        all_rows.setdefault(tool, []).extend(dict(r, sample=sample) for r in rows)
        all_contigs.setdefault(tool, []).extend(dict(r, sample=sample) for r in crow)
        _, prec = summarize(rows, crow)
        precision_rows.append({"tool": tool, "sample": sample, **prec})
        print(tool, sample, "done", flush=True)


## Доля точно восстановленных V…J (`exact_full`) по классам восстановимости

In [ ]:
import pandas as pd
summary = []
for tool, rows in all_rows.items():
    for keys in (("reconstructability",), ("copies",), ("locus",)):
        s, _ = summarize(rows, [], keys)
        for r in s:
            summary.append({"tool": tool, "group_by": keys[0], "group": r[keys[0]],
                            **{k: v for k, v in r.items() if k != keys[0]}})
    for r in summarize_clonotypes(rows):
        summary.append({"tool": tool, "group_by": "cdr3_clonotypes", "group": f"{r['locus']}:{r['clonotype_state']}",
                        "templates": r["clonotypes"], "cdr3_in_contig_frac": r["cdr3_in_contig_frac"]})
BENCH_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary).to_csv(BENCH_DIR / "assemblers_summary.tsv", sep="\t", index=False)
pd.DataFrame(precision_rows).to_csv(BENCH_DIR / "assemblers_precision.tsv", sep="\t", index=False)
df = pd.DataFrame(summary)
df[df.group_by == "reconstructability"].pivot_table(index="group", columns="tool", values="exact_full_frac")


# Структура контигов по поддержке ридами и пороги отсечения по поддержке
BINS = [(1, 1, "1"), (2, 4, "2-4"), (5, 9, "5-9"), (10, 49, "10-49"), (50, float("inf"), "50+")]
THRESHOLDS = (1, 2, 3, 5, 10, 20, 50)
struct_rows, thr_rows = [], []
for tool, crows in all_contigs.items():
    sup = {(r["sample"], r["contig_id"]): float(r["support"]) for r in crows if str(r.get("support", "")) not in ("", "nan")}
    if not sup:
        continue
    kind = "reads" if tool == "trust4" else "kmer_coverage"
    for lo, hi, lab in BINS:
        rs = [r for r in crows if lo <= sup.get((r["sample"], r["contig_id"]), -1) <= hi]
        c = Counter(r["structure"] for r in rs)
        struct_rows.append({"tool": tool, "support_type": kind, "support_bin": lab, "contigs": len(rs),
                            **{k: round(c[k] / max(1, len(rs)), 4) for k in STRUCTURE}})
    ver = [r for r in all_rows[tool] if str(r["verified"]) == "True"]
    for t in THRESHOLDS:
        kept = [r for r in crows if sup.get((r["sample"], r["contig_id"]), -1) >= t]
        c = Counter(r["structure"] for r in kept)
        exact = sum(1 for r in ver if any(sup.get((r["sample"], c), -1) >= t for c in str(r.get("exact_contigs", "")).split(";") if c))
        thr_rows.append({"tool": tool, "support_type": kind, "min_support": t, "contigs_kept": len(kept),
                         "verified_exact_recall": round(exact / max(1, len(ver)), 4),
                         **{k: round(c[k] / max(1, len(kept)), 4) for k in STRUCTURE}})
pd.DataFrame(struct_rows).to_csv(BENCH_DIR / "assemblers_structure_by_support.tsv", sep="\t", index=False)
pd.DataFrame(thr_rows).to_csv(BENCH_DIR / "assemblers_support_thresholds.tsv", sep="\t", index=False)
display(pd.DataFrame(struct_rows)); display(pd.DataFrame(thr_rows))


## CDR3 по клонотипам (прочитанные и контроль)

In [ ]:
df[df.group_by == "cdr3_clonotypes"].pivot_table(index="group", columns="tool", values="cdr3_in_contig_frac")


## Precision контигов

In [ ]:
pd.DataFrame(precision_rows)


## Блоки фазы

In [ ]:
# Блоки фазы: точность блоков, ложные склейки, покрытие эталона правильными блоками
phase_rows = []
for tool, rows in all_rows.items():
    s, p = summarize(rows, all_contigs[tool])
    allrow = next(r for r in s if r["reconstructability"] == "ALL")
    phase_rows.append({"tool": tool, **{k: v for k, v in p.items() if k.startswith(("block_", "phase_", "contigs_with", "switches", "novel"))},
                       **{k: v for k, v in allrow.items() if k.startswith("exact_block") or k == "exact_full_frac"}})
pd.DataFrame(phase_rows).to_csv(BENCH_DIR / "assemblers_phase_blocks.tsv", sep="\t", index=False)
pd.DataFrame(phase_rows)